# CDC 2024 NICU-admission research baseline
Step-by-step starter for Google Colab. **Not a clinical prediction tool and not connected to the dashboard.**

Target: NICU admission **recorded on the birth certificate**, not death, genetic disease or general newborn health. A NICU admission is affected by access and clinical practice. No event timestamps are available to establish a prospective prediction at five minutes; treat this as retrospective classification until timing is verified.

Use public U.S. **singleton** births only, with exact-coded maternal age 13–49, obstetric gestation 20–45 weeks and birth weight 227–5999 g. These exclusions define a restricted research population, not clinical normality. Ages 12 and 50 are grouped/top-coded in the file and are excluded.

We omit APGAR and recorded interventions because their timing relative to NICU admission is unclear. The dataset has APGAR totals, not the five components; do not fabricate components or a generic delivery-complication flag. This model does not validate the current fuzzy index, family-history module or safety overrides.

Sources: [CDC portal](https://www.cdc.gov/nchs/data_access/vitalstatsonline.htm), [2024 guide](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/DVS/natality/UserGuide2024.pdf), [scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html).

**Before running:** read the CDC data-user agreement linked from the portal. This notebook exports a research artifact, not a deployment-ready model.


## 1. Start a CPU runtime and install dependencies
Open this notebook in Colab and run cells from top to bottom. No GPU is needed. Versions are recorded in the exported metadata; repeatability also depends on retaining the exact ZIP and its checksum.


In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib joblib
!apt-get -qq update
!apt-get -qq install -y p7zip-full


In [ ]:
from pathlib import Path
import io, json, hashlib, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 42
MAX_SAMPLE = 120_000  # Uniform sample across the entire eligible file, not its first rows.
FEATURES = ['gestational_age_weeks', 'birth_weight_g', 'maternal_age']
OUTPUT = Path('/content/cdc_nicu_results')
OUTPUT.mkdir(exist_ok=True)

import zipfile


## 2. Obtain the 2024 U.S. natality ZIP
Default: download the official **Nat2024us.zip**, about 232 MB compressed. Parsing scans the complete ZIP without extracting its much larger raw file and can take several minutes. If the CDC server cannot be reached from Colab, set `DATA_MODE='upload'`, download the U.S. 2024 ZIP from the CDC portal yourself and select it when prompted. Do not upload a CSV, Territories ZIP or another year: those require a separately verified layout.


In [ ]:
DATA_MODE = 'download'  # 'download' or 'upload'
DATA_URL = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/DVS/natality/Nat2024us.zip'
if DATA_MODE == 'download':
    import urllib.request
    zip_path = Path('/content/Nat2024us.zip')
    urllib.request.urlretrieve(DATA_URL, zip_path)
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one 2024 U.S. natality ZIP.')
    filename, content = next(iter(uploaded.items()))
    zip_path = Path('/content') / Path(filename).name
    zip_path.write_bytes(content)
    del uploaded, content
if zip_path.name.lower() != 'nat2024us.zip':
    raise ValueError('Use Nat2024us.zip: this parser is verified only against the 2024 guide.')
hash_digest = hashlib.sha256()
with zip_path.open('rb') as stream:
    for block in iter(lambda: stream.read(1024*1024), b''):
        hash_digest.update(block)
zip_sha256 = hash_digest.hexdigest()
print('ZIP SHA-256:', zip_sha256)


## 3. Parse, exclude unknown/non-reporting labels, and sample uniformly
CDC positions are **1-based, inclusive**; Python slices are **0-based, end-exclusive**.

| Item | CDC positions | Notes |
|---|---|---|
| MAGER | 75–76 | 12 groups ages 10–12; 50 means 50+; retain 13–49 |
| DPLURAL | 454 | Retain 1 (singleton) |
| OEGest_Comb | 499–500 | 99 unknown; retain 20–45 |
| DBWT | 504–507 | 9999 unknown; retain 227–5999 g |
| AB_NICU | 519 | Y / N / U; unknown is **not** No |
| F_AB_NIUC | 526 | Guide's spelling; 1 reporting, 0 non-reporting |

Use each item's reporting flag, not a guessed label. The scope below excludes the grouped ages, out-of-domain values and unknown targets rather than imputing outcomes. Record exclusion rates: this restriction can bias the study population.


In [ ]:
COLS = ['maternal_age','plurality','gestational_age_weeks','birth_weight_g','nicu_raw','nicu_reporting']
COLSPECS = [(74,76),(453,454),(498,500),(503,507),(518,519),(525,526)]

def clean_chunk(raw):
    df = raw.copy()
    for column in ['maternal_age','plurality','gestational_age_weeks','birth_weight_g']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    known_label = df.nicu_raw.isin(['Y','N']) & df.nicu_reporting.eq('1')
    in_domain = (df.plurality.eq(1) & df.maternal_age.between(13,49)
                 & df.gestational_age_weeks.between(20,45)
                 & df.birth_weight_g.between(227,5999))
    result = df.loc[known_label & in_domain, FEATURES].copy()
    result['nicu_admission'] = df.loc[result.index,'nicu_raw'].map({'N':0,'Y':1}).astype(int)
    counts = {'raw':len(df), 'unknown_or_nonreporting_target':int((~known_label).sum()),
              'outside_feature_or_singleton_scope':int((~in_domain).sum()), 'eligible':len(result)}
    # Exclusion reason counts overlap; eligible is the joint intersection.
    return result, counts

# Parser/cleaning self-test using an invented fixed-width row, not patient data.
def fixture(nicu='Y', flag='1', gest='39', weight='3200', age='29', plurality='1'):
    line = list(' ' * 600)
    for (start,end),value in zip(COLSPECS,[age,plurality,gest,weight,nicu,flag]):
        line[start:end] = list(value)
    return ''.join(line)+'\n'
fixture_raw = pd.read_fwf(io.StringIO(fixture()+fixture(nicu='U')+fixture(flag='0')+fixture(age='50')),
                          colspecs=COLSPECS, names=COLS, header=None, dtype=str)
fixture_clean, fixture_counts = clean_chunk(fixture_raw)
assert len(fixture_clean)==1 and fixture_clean.iloc[0]['birth_weight_g']==3200
assert fixture_clean.iloc[0]['nicu_admission']==1
print('Fixed-width parser self-test: passed')


In [ ]:
import io, shutil, subprocess, tempfile, zipfile
from contextlib import contextmanager

@contextmanager
def open_zip_text(zip_path, member_name):
    executable = shutil.which('7z') or shutil.which('7zz')
    if executable is None:
        raise RuntimeError('Install p7zip-full in Colab before parsing.')
    # Stream decompressed records; stderr goes to disk to avoid a full pipe blocking.
    with tempfile.TemporaryFile() as errors:
        process = subprocess.Popen(
            [executable, 'x', '-so', '-bd', '-y', '-spd', str(zip_path), member_name],
            stdout=subprocess.PIPE, stderr=errors)
        try:
            with io.TextIOWrapper(process.stdout, encoding='latin-1') as text:
                yield text
            returncode = process.wait()
            if returncode != 0:
                errors.seek(0)
                details = errors.read().decode('utf-8', errors='replace')
                raise RuntimeError(f'7-Zip failed (exit {returncode}); discard this partial sample. {details[-4000:]}')
        finally:
            if process.poll() is None:
                process.kill()
                process.wait()
            if process.stdout is not None:
                process.stdout.close()

globals().pop('df', None)  # A failed rerun must not leave an old training dataframe.
rng = np.random.default_rng(SEED)
sample = pd.DataFrame()
audit = {'raw':0,'unknown_or_nonreporting_target':0,'outside_feature_or_singleton_scope':0,'eligible':0}
with zipfile.ZipFile(zip_path) as archive:
    members = [m for m in archive.infolist() if not m.is_dir() and m.file_size > 100_000_000]
    if len(members)!=1:
        raise ValueError('Expected exactly one large U.S. natality fixed-width file; inspect ZIP before proceeding.')
    member = members[0]
    print('ZIP compression method:', member.compress_type)
    print('Parsing:', member.filename, 'uncompressed bytes:', member.file_size)
    with open_zip_text(zip_path, member.filename) as text:
        for number, raw in enumerate(pd.read_fwf(text, colspecs=COLSPECS, names=COLS,
                                                 header=None, dtype=str, chunksize=50_000), 1):
            eligible, counts = clean_chunk(raw)
            for key,value in counts.items(): audit[key] += value
            eligible['_sample_key'] = rng.random(len(eligible))
            # Keeping the smallest independent random keys is a uniform sample across all eligible rows.
            sample = pd.concat([sample,eligible], ignore_index=True).nsmallest(MAX_SAMPLE,'_sample_key')
            if number%10==0: print('Rows scanned:',audit['raw'],'eligible:',audit['eligible'])
df = sample.drop(columns='_sample_key').reset_index(drop=True)
if len(df)<1000 or df.nicu_admission.nunique()!=2:
    raise ValueError('Insufficient valid two-class data: verify file, layout and exclusions.')
print('Audit (reason counts overlap):',audit)
print('Sample:',len(df),'NICU prevalence:',df.nicu_admission.mean())
df.describe()


## 4. Split before fitting any transformation
60% training, 20% validation, 20% untouched test, stratified by the outcome. Singleton restriction avoids splitting twins from the same birth across sets. Same-year random holdout is **internal validation only**; it does not replace temporal or external hospital validation. No hyperparameter or threshold is chosen on test data.


In [ ]:
from sklearn.model_selection import train_test_split
X, y = df[FEATURES], df.nicu_admission
X_train, X_hold, y_train, y_hold = train_test_split(X,y,test_size=.4,stratify=y,random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_hold,y_hold,test_size=.5,stratify=y_hold,random_state=SEED)
print('Train/validation/test:',len(X_train),len(X_val),len(X_test))


## 5. Fit an interpretable baseline and calibrate within training folds
Scaling is fitted only within training folds. No class balancing or outcome-driven oversampling is used, preserving the sampled outcome prevalence. Logistic regression models the three birth measurements; it has no knowledge of APGAR components or the current binary complication input. The sigmoid calibration uses training-fold data only. This is a first baseline, not an assertion that three features suffice clinically.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
model = CalibratedClassifierCV(baseline, method='sigmoid', cv=3)
model.fit(X_train,y_train)
dummy = DummyClassifier(strategy='prior').fit(X_train,y_train)


## 6. Review validation metrics, without claiming a clinical threshold
Report AUROC and AUPRC for discrimination, Brier score and a reliability plot for probability behaviour, plus sensitivity/specificity/PPV at a **demonstration-only** threshold 0.5. For a rare outcome, AUPRC must be compared with prevalence. Brier score alone does not establish calibration. A clinical action threshold needs validation-set analysis and clinical approval; these thresholds do not correspond to the fuzzy Low/Moderate/High cutoffs.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
from sklearn.calibration import calibration_curve
THRESHOLD = .5  # Research demonstration only; not an action threshold.
def metrics(y,p):
    tn,fp,fn,tp = confusion_matrix(y,p>=THRESHOLD,labels=[0,1]).ravel()
    def ratio(a,b): return float(a/b) if b else None
    return {'n':len(y),'prevalence':float(np.mean(y)),
            'auroc':float(roc_auc_score(y,p)), 'auprc':float(average_precision_score(y,p)),
            'brier':float(brier_score_loss(y,p)), 'threshold':THRESHOLD,
            'sensitivity':ratio(tp,tp+fn),'specificity':ratio(tn,tn+fp),'ppv':ratio(tp,tp+fp),
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
validation_metrics = metrics(y_val,model.predict_proba(X_val)[:,1])
print('Validation:',validation_metrics)


## 7. Evaluate the untouched test set once and check subgroup performance
The test set is locked for final reporting; after using it to redesign the model, it is no longer an untouched test. Compare against the prevalence-only baseline. Subgroups below are descriptive gestation/age/weight checks, not a full fairness assessment. Small or one-class groups cannot support reliable comparisons. For presentation-quality work, add bootstrap confidence intervals, temporal validation on a separately verified year's layout, reporting/completeness analysis and local external validation.


In [ ]:
p_test = model.predict_proba(X_test)[:,1]
test_metrics = metrics(y_test,p_test)
dummy_metrics = metrics(y_test,dummy.predict_proba(X_test)[:,1])
print('Untouched test:',test_metrics)
print('Prevalence baseline:',dummy_metrics)
observed,predicted = calibration_curve(y_test,p_test,n_bins=10,strategy='quantile')
plt.plot([0,1],[0,1],'--',color='gray'); plt.plot(predicted,observed,'o-')
plt.xlabel('Predicted probability of recorded NICU admission'); plt.ylabel('Observed proportion')
plt.title('Internal test reliability diagram'); plt.tight_layout()
plt.savefig(OUTPUT/'calibration.png',dpi=160); plt.show()
subgroups = {'preterm':X_test.gestational_age_weeks<37,'term_or_later':X_test.gestational_age_weeks>=37,
             'low_weight':X_test.birth_weight_g<2500,'not_low_weight':X_test.birth_weight_g>=2500,
             'maternal_age_35plus':X_test.maternal_age>=35,'maternal_age_under35':X_test.maternal_age<35}
subgroup_metrics = {}
for name,mask in subgroups.items():
    yy=y_test.loc[mask]; pp=p_test[np.asarray(mask)]
    subgroup_metrics[name] = metrics(yy,pp) if len(yy)>=200 and yy.nunique()==2 else {'n':len(yy),'status':'insufficient data'}
print(json.dumps(subgroup_metrics,indent=2))


## 8. Export research artifacts and record limitations
Exporting does not integrate this model into the dashboard. Retain all metadata and compare performance with the prevalence baseline. Do not use a downloaded model file from an untrusted source: joblib loading executes Python objects. The next integration stage requires a local outcome dataset, validated prediction timing, approved thresholds and a model card. The safety override must remain independent of an ML probability.


In [ ]:
import joblib
metadata = {'status':'research_only_not_clinically_validated','dataset':'CDC 2024 U.S. natality',
            'guide':'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/DVS/natality/UserGuide2024.pdf',
            'zip_sha256':zip_sha256,'target':'AB_NICU Y vs N, reporting flag at position 526 equals 1',
            'prediction_timing':'retrospective birth-record classification; prospective timing not verified',
            'features':FEATURES,'population':'singleton, gestation 20–45 weeks, weight 227–5999g, exact-coded maternal age 13–49',
            'split':'same-year internal 60/20/20 stratified','random_seed':SEED,'sample_limit':MAX_SAMPLE,
            'exclusion_audit':audit,'validation_metrics':validation_metrics,'test_metrics':test_metrics,
            'baseline_test_metrics':dummy_metrics,'subgroup_metrics':subgroup_metrics,
            'versions':{package:importlib.metadata.version(package) for package in ['numpy','pandas','scikit-learn','joblib']},
            'limitations':['No local or temporal external validation','No individual APGAR components or family-history fields',
                           'No reliable event ordering for a five-minute prediction','Restricted complete-case population',
                           'NICU reflects care access and practice as well as health','No clinical action threshold approved']}
joblib.dump(model,OUTPUT/'nicu_research_model.joblib')
(OUTPUT/'model_card.json').write_text(json.dumps(metadata,indent=2))
import shutil
bundle=shutil.make_archive('/content/cdc_nicu_research_results','zip',OUTPUT)
from google.colab import files
files.download(bundle)
print('Research export complete. No clinical deployment has occurred.')
